# Atlas-WX data: every qualifying ISD station, 1995–2025
Runs `atlas/wx/fetch_isd_all.py` (rules: `atlas/PREREG_WX.md` §1 and its addendum). Raw files go to the runtime's local disk; the packed yearly files, the frozen station list and its SHA-256 go to Drive (`PWM-WX/isd`). No account or key is needed.

Run cells 1–2 first (a 200-station trial, ~5 min). If it ends with a `WX-DATA` line, run cell 3 (the full fetch, several hours, in the background) and check it with cell 4.

In [ ]:
# 1. Drive and code
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/pwm'):
    !git clone -q -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model /content/pwm
!cd /content/pwm && git pull -q && git log --oneline -1
!df -h /content | tail -1

In [ ]:
# 2. Trial on the first 200 candidate stations (writes to a separate trial folder)
!cd /content/pwm/atlas/wx && python -u fetch_isd_all.py --raw /content/isd_raw --out /content/drive/MyDrive/PWM-WX/isd_trial --limit 200

In [ ]:
# 3. Full fetch in the background (resumable: re-run this cell after a disconnect)
!mkdir -p /content/drive/MyDrive/PWM-WX/isd
!cd /content/pwm/atlas/wx && nohup python -u fetch_isd_all.py --raw /content/isd_raw --out /content/drive/MyDrive/PWM-WX/isd > /content/drive/MyDrive/PWM-WX/isd/fetch.log 2>&1 &
print('started; check with cell 4')

In [ ]:
# 4. Progress
!tail -5 /content/drive/MyDrive/PWM-WX/isd/fetch.log; ls -la /content/drive/MyDrive/PWM-WX/isd | tail -5; du -sh /content/isd_raw 2>/dev/null

## Stage 1 (after the data step has finished; a GPU runtime with high RAM, ideally A100)
Each cell is resumable: if the session ends, run the same cell again. Order: 10M, then 30M and 100M grown from it. Results go to `PWM-WX/runs`.

In [ ]:
# 5. Stage 1, 10M
!cd /content/pwm && git pull -q
!mkdir -p /content/drive/MyDrive/PWM-WX/runs
!cd /content/pwm/atlas/wx && nohup python -u run_atlas_wx.py --size 10000000 --tag wx_s1_10m --pause-at-cap --cap 600 > /content/drive/MyDrive/PWM-WX/runs/log_wx_s1_10m.txt 2>&1 &

In [ ]:
# 6. Stage 1, 30M and 100M (grown from the trained 10M; run after cell 5 has written wx_s1_10m_s600.json)
R='/content/drive/MyDrive/PWM-WX/runs'
!cd /content/pwm/atlas/wx && nohup sh -c "python -u run_atlas_wx.py --init $R/wx_s1_10m_s600.pt --grow 1,7,7 --lr 3e-4 --tag wx_s1_30m --pause-at-cap --cap 600 > $R/log_wx_s1_30m.txt 2>&1; python -u run_atlas_wx.py --init $R/wx_s1_10m_s600.pt --grow 2,6,6 --lr 3e-4 --tag wx_s1_100m --pause-at-cap --cap 600 > $R/log_wx_s1_100m.txt 2>&1" &

In [ ]:
# 7. Progress
!tail -3 /content/drive/MyDrive/PWM-WX/runs/log_wx_s1_*.txt; nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv